ITT:  
1. [Connecting to LLM](#connecting-llm)  
2. [Running Sandboxed Environment](#running-sandboxed-environment)  
3. [Testing on MBPP](#testing-on-mbpp)  
    


# Connecting LLM

> [!NOTE]  
> Please don't abuse these services, else we might lose them.

Go to [link](https://huggingface.co/settings/tokens) to regenerate HF_TOKEN.

In [6]:
# %env HF_TOKEN=asd

In [ ]:
# %env GEMINI_API_KEY=asd

In [4]:
%env LLM_PROVIDER=huggingface

env: LLM_PROVIDER=huggingface


In [ ]:
# Set HF_TOKEN for this notebook session (use your token from https://huggingface.co/settings/tokens)
# Option 1: Jupyter magic (persists to all cells)
# %env HF_TOKEN=your_token_here

# Option 2: Python (persists to all cells)
import os
os.environ["HF_TOKEN"] = os.environ.get("HF_TOKEN", "your_token_here")
# print(os.environ["HF_TOKEN"])

In [ ]:
# %uv pip install openai

In [ ]:
from google import genai

# The client gets the API key from the environment variable `GEMINI_API_KEY`.
client = genai.Client()

response = client.models.generate_content(
    model="gemini-2.5-flash-lite", contents="Tell me a joke lulW"
)

print(response.text)

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=os.environ["HF_TOKEN"],
)

def get_completion(prompt):
    completion = client.chat.completions.create(
        model="moonshotai/Kimi-K2-Instruct-0905",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ],
)
    return completion.choices[0].message.content



In [ ]:
# Test Qwen via Ollama (run: ollama serve && ollama pull qwen2.5:0.5b)
from llm.provider import get_client, get_provider_config

client = get_client(provider="ollama")
cfg = get_provider_config(provider="ollama")
print(f"Model: qwen2.5:7b")

response = client.chat.completions.create(
    model="llama3.1:8b",
    messages=[{"role": "user", "content": "Do you think there is somehting like the objectively best knock knock joke?"}],
)
print(response.choices[0].message.content)

In [ ]:
print(get_completion("Tell me a joke"))

---- Test done ----

# Running Sandboxed Environment

> We decided to explore Firecracker, docker and microVMs and decided to use Docker for now as Firecracker is overkill

Build the image (from `agent-smith/`):

```bash
docker build -f DOCKER/Dockerfile -t my-agent-image .
```

Use:
```bash
uv run --active sandbox
```

In [ ]:
!docker run --rm \
  --memory=512m \
  --cpus=1 \
  --pids-limit=128 \
  --read-only \
  --cap-drop=ALL \
  --security-opt=no-new-privileges \
  --network=none \
  my-agent-image


Use:
```bash
uv run --active sandbox sandbox_template.json
```

```json
// sandbox_template.json

```

Use:  
```bash
uv run sandbox --mcp-server <URL>
```


# Testing on MBPP

> Assignment wants to have tests. 

```bash
# 1. Dump a task
cd moulinette_master
uv run moulinette_eval dump mbpp --output ../cache/mbpp_task.json
# 2. Run your agent
cd ../student
uv run python -m agent_mbpp --task-file ../cache/mbpp_task.json \
--output ../cache/mbpp_solution.json
# 3. Validate solution
cd ../moulinette_master
uv run moulinette_eval validate mbpp ../cache/mbpp_task.json \
../cache/mbpp_solution.json
```

Testing on SWE

```bash
# 1. Dump a task (from agent-smith/)
cd moulinette_master
uv run moulinette_eval dump swebench --output ../cache/swebench_task.json
# 2. Run your agent (from student/)
cd ../student
uv run python -m agent_swebench --task-file ../cache/swebench_task.json --output ../cache/swebench_solution.json
# 3. Validate solution
cd moulinette_master
uv run moulinette_eval validate swebench ../cache/swebench_task.json ../cache/swebench_solution.json
```

# Building Agents
#### BUILDING Agent_mbpp

Agents Parts:  
1. [mbpp agent](#building-agent_mbpp)
1. [swe agent](#building-agent_swe)

autonomous agent dedicated to solving Mostly Basic Python Problems

### 1. Dump a random task

In [1]:
!cd moulinette_master && uv run moulinette_eval dump mbpp --output ../cache/mbpp_task.json

Task 475 dumped to: ../cache/mbpp_task.json
Task saved to: ../cache/mbpp_task.json


### 2. Run the agent on task

In [7]:
!cd student && uv run python -m agent_mbpp --task-file ../cache/mbpp_task.json --output ../cache/mbpp_solution.json


TASK
Task ID: 475

Task: Write a function to sort counter by value.

Function:
def sort_counter(dict1):

Tests:
assert sort_counter({'Math':400, 'Physics':300, 'Chemistry':250})==[('Math', 400), ('Physics', 300), ('Chemistry', 250)]
assert sort_counter({'Math':900, 'Physics':1000, 'Chemistry':1250})==[('Chemistry', 1250), ('Physics', 1000), ('Math', 900)]


--- Iteration 1 ---

SOLUTION:
def sort_counter(dict1):
    if not dict1:
        return []
    return sorted(dict1.items(), key=lambda x: x[1], reverse=True)

Tests: PASSED

Wrote ../cache/mbpp_solution.json
Success: True


### 3. Validate the solution

In [8]:
!cd moulinette_master && uv run moulinette_eval validate mbpp ../cache/mbpp_task.json ../cache/mbpp_solution.json


VALIDATING SOLUTION
Task ID: 475
Benchmark: mbpp
Success claimed: True

STEP 1: CORRECTNESS VALIDATION
Correctness: PASSED

STEP 2: METRICS VALIDATION
Iterations: 1 / 10 OK
Input tokens: 177 / 4000 OK
Output tokens: 34 / 1000 OK
Time: 7.2s / 60.0s OK
Metrics: VALID

FINAL RESULT
Correctness: PASSED
Metrics: VALID
Overall: PASSED


```
Task loading  
Agent execution  
```

### BUILDING agent_swe

> This part focuses on implementing an autonomous agent capable of solving SWE-bench tasks inside Dockerized environments.  
> Usages:  
>   1. Fix real bugor implement features in real repositories
>   2. Explore codebases inside Docker containers, you are responsible to clean it after your program execution
>   3. Generate and submit valid patches using ’git -c core.fileMode=false diff’

```python
#task input
class SWEBenchTaskInput(BaseModel):
    """Input for SWE-bench task evaluation.
    You are responsible for pulling and managing the Docker container.
    The docker_image field contains the full image name to pull.
    The eval_script is used to run tests inside the container.
    """
    instance_id: str
    repo: str = ""
    docker_image: str # Full image name, e.g., "swebench/sweb.eval.x86_64.
    sympy_1776_sympy-23534:latest"
    problem_statement: str
    hints_text: str = ""
    eval_script: str # Bash script to run tests inside the container
```

The bellow part contains a "patch"...

##### EXAMPLE:

SWE-bench is a benchmark for evaluating autonomous coding agents. It tests whether an agent can fix real GitHub issues from real repositories.

- It must be a valid code change  
- Written as a unified diff (`diff` / git diff format)  
- Modifies the repository files  
- No extra commentary — just the patch  

Example structure:
```diff
diff --git a/file.py b/file.py
index 123..456 100644
--- a/file.py
+++ b/file.py
@@ -10,7 +10,7 @@
-    return x + 1
+    return x + 2
```

#### class templates:

```python
#agent output
class StepMetrics(BaseModel):
    """Metrics for a single agent step."""
    step: int
    input_tokens: int
    output_tokens: int
    request_time_ms: float
    timestamp: str = Field(default_factory=lambda: datetime.now().
    isoformat())

class SolutionOutput(BaseModel):
    """Result of your solution: this is what you need to produce."""
    task_id: str
    benchmark: str # "mbpp" or "swebench"
    success: bool
    solution: str # Code for MBPP, patch for SWE-bench
    iterations: int
    total_requests: int
    total_input_tokens: int
    total_output_tokens: int
    total_time_seconds: float
    steps: List["StepMetrics"] = Field(default_factory=list)
    error: Optional[str] = None
    timestamp: str = Field(default_factory=lambda: datetime.now().
    isoformat())
```

### 1. Dump a random task

In [59]:
!cd moulinette_master && uv run moulinette_eval dump swebench --output ../cache/swebench_task.json

2026-02-21 23:40:25,766 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/SWE-bench/SWE-bench_Verified/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-02-21 23:40:25,776 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/SWE-bench/SWE-bench_Verified/99450355ca8c611021187a57ffac304b66666738/README.md "HTTP/1.1 200 OK"
2026-02-21 23:40:25,896 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/SWE-bench/SWE-bench_Verified/resolve/99450355ca8c611021187a57ffac304b66666738/SWE-bench_Verified.py "HTTP/1.1 404 Not Found"
2026-02-21 23:40:26,263 - httpx - INFO - HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/SWE-bench/SWE-bench_Verified/SWE-bench/SWE-bench_Verified.py "HTTP/1.1 404 Not Found"
2026-02-21 23:40:26,388 - httpx - INFO - HTTP Request: GET https://huggingface.co/api/datasets/SWE-bench/SWE-bench_Verified/revision/99450355ca8c611021187a57ffac304b66666738 "HTTP/1.1 

### 2. Run the agent on task

In [60]:
!cd ./student && uv run python -m agent_swebench --task-file ../cache/swebench_task.json --output ../cache/swebench_solution.json


SWE-BENCH TASK
Instance: sympy__sympy-23534
Docker image: swebench/sweb.eval.x86_64.sympy_1776_sympy-23534:latest
Test: SWE-bench eval script (setup + run tests; each run_tests re-runs the test block)

Problem:
Using symbols to create functions doesn't work if there is an extra layer of parentheses
Sympy version == 1.10.1

Using `symbols` to create symbol-like objects like instances of `Function` as shown in the [documentation](https://docs.sympy.org/latest/modules/core.html?highlight=symbols#symbols) creates objects of class `Symbol` instead of `Function` if there is an extra layer of parentheses.

The extra layer of parentheses are necessary to deconstruct the output as separate tuples.

Runnin...

Pulling Docker image...
Starting container...
Starting MCP server in container...
MCP server at http://localhost:8766

--- Iteration 1 ---
  Calling LLM...
  write_file: ok

--- Iteration 2 ---
  Calling LLM...
  find_relevant: ok

--- Iteration 3 ---
  Calling LLM...
  search_symbol: ok


### 3. Validate the task completion

In [58]:
!cd ./moulinette_master && uv run moulinette_eval validate swebench ../cache/swebench_task.json ../cache/swebench_solution.json


VALIDATING SOLUTION
Task ID: sympy__sympy-15875
Benchmark: swebench
Success claimed: True

STEP 1: CORRECTNESS VALIDATION
Started container 77aec9892faf316cc7ec49e02bde2e13d16ceab9a6ca2704b3a4c7caa49f7bcc
2026-02-21 23:39:55,354 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/SWE-bench/SWE-bench_Verified/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-02-21 23:39:55,365 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/SWE-bench/SWE-bench_Verified/99450355ca8c611021187a57ffac304b66666738/README.md "HTTP/1.1 200 OK"
2026-02-21 23:39:55,487 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/SWE-bench/SWE-bench_Verified/resolve/99450355ca8c611021187a57ffac304b66666738/SWE-bench_Verified.py "HTTP/1.1 404 Not Found"
2026-02-21 23:39:55,841 - httpx - INFO - HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/SWE-bench/SWE-bench_Verified/SWE-bench/SWE-bench_Verified.py "HT

--------------------- footer ---------------------

Leveraging:
1. uv for package managing
2. hugging face for api
    - this depleted, so I will try to move to [this google thing](https://aistudio.google.com/api-keys?project=gen-lang-client-0361947210)  
3. openai module